In [38]:
stock_symbol = "tsla" #e.g. tsla, msft
column = 'content' # which column to clean

file = f"Data/Stocktwits/{stock_symbol}_posts.csv"

output_filename = f"{stock_symbol.upper()}_cleaned.csv"

# Combine csv files (if needed)

'''ensure that all files csv files you want to combine are in the same folder '''

import os
import glob
import pandas as pd

path = '#path'
extension = 'csv'
os.chdir(path)
output_filename = path.split('/')[-1] + '_combined.csv'

all_files = [i for i in glob.glob('*.{}'.format(extension))]

combined_csv = pd.concat([pd.read_csv(f) for f in all_files])
combined_csv.to_csv(output_filename, index = False, encoding = 'utf-8-sig')

# Cleaning text

In [15]:
import re
import nltk
import numpy as np
import pandas as pd
from nltk import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\wordnet.zip.


True

In [16]:
# text cleaning
def text_lower(text):
    return ' '.join([t.lower() for t in text.split()])

def remove_hashtag_mentions_urls(text):
    return re.sub(r"(?:\@|\#|https?\://)\S+", "", text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  # emoticons
    u"\U0001F300-\U0001F5FF"  # symbols & pictographs
    u"\U0001F680-\U0001F6FF"  # transport & map symbols
    u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

    return emoji_pattern.sub(r'', text)

def tokenization(text):
#     word_tokenizer = RegexpTokenizer(r'[-\'\w]+')
    word_tokenizer = RegexpTokenizer(r'\w+')
    tokenized_text = word_tokenizer.tokenize(text)
    return tokenized_text

def remove_numbers(text):
    return [t for t in text if not t.isdigit()]

def remove_stopwords(text, symbol):
    word_dictionary = {
        'aapl': ['$AAPL', 'apple'],
        'googl': ['$GOOGL', 'google'],
        'msft': ['$MSFT', 'microsoft'],
        'tsla': ['$TSLA', 'tesla'],
        'amzn': ['$AMZN', 'amazon'],
        'vxrt': ['$VXRT', 'vaxart'],
        'rui': ['$RUI', 'russell', 'index'],
        'ark': ['$ARK', 'resources'],
        'ater': ['$ATER', 'Aterian']
    }
    stop_words = stopwords.words('english')
    stop_words.append(symbol)
    
    try:
        for word in word_dictionary[symbol]:
            stop_words.append(word)
    except:
        pass 
        
    text_stopremoved = [w for w in text if w not in stop_words]
    return text_stopremoved

def lemmatization(text):
    lemma_word = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for w in text:
        word1 = wordnet_lemmatizer.lemmatize(w, pos = "n")
        word2 = wordnet_lemmatizer.lemmatize(word1, pos = 'v')
        word3 = wordnet_lemmatizer.lemmatize(word2, pos = ('a'))
        lemma_word.append(word3)
    return ' '.join(lemma_word)
    

def clean_text(text, symbol):
    text = text_lower(text)
    text = remove_hashtag_mentions_urls(text)
    text = remove_emoji(text)
    text = tokenization(text)
    text = remove_numbers(text)
    text = remove_stopwords(text, symbol)
    text = lemmatization(text)
    return text 

# Comment Classifier

In [17]:
def comment_classifier(file, column, stock_symbol): 
    df = pd.read_csv(file)
    
    #cleaning text
    text = df[column].to_list()
    cleaned_text = [clean_text(t, stock_symbol) if len(t) > 0 else '' for t in text]
    
    output = 'cleaned ' + column
    df[output] = cleaned_text
    
    #remove missing values from updated df 
    df[output].replace('', np.nan, inplace = True)
    df.dropna(subset=[output], inplace = True)
    
    #for twitter
    try:
        #df = df[df[output].apply(lambda x: len(x)>4)]
        df = df[df['Number of Retweets'] > 5]
        df = df[df['Number of Likes'] > 200]
        df = df[df['Number of Followers'] > 50]
        df = df[df[output].str.split().str.len() > 4]

    except:
        pass
    
    
    return df

In [39]:
df = comment_classifier(file, column, stock_symbol)

df.head(3)

,person_id,content,date_created,time_created,username,name,join_date,official,followers,upload_source,likes,compound_score,pos_score,neg_score,neu_score,cleaned content
0,389607047,$TSLA FSD = crashes. for the car and for the s...,2021-10-11,12:34:16,Nancypelosi0,Obi.,2021-08-12,False,0,StockTwits for iOS,1,NaN,NaN,NaN,NaN,fsd crash car stock
1,389606777,$TSLA TSLA 2021-10-10 Daily Forecast Video: \n...,2021-10-11,12:33:29,allcharts,Magnus Sarkozy,2019-11-08,False,1660,Test01,0,NaN,NaN,NaN,NaN,daily forecast video
2,389606686,Seems $PTGX and $CCXI would be following he s...,2021-10-11,12:33:11,THWAYINVTR,Throwaway Investor,2020-06-08,False,7,StockTwits for iOS,0,NaN,NaN,NaN,NaN,seem ptgx ccxi would follow strategy move towa...


In [40]:
df.to_csv(output_filename)